# Cellpose fine-tuned: maschere + probabilità su train, val e test (Kaggle)

Serve per il confronto alla pari "Cellpose fine-tuned + classificatore" contro "v2 + classificatore": il classificatore si
addestra sugli oggetti del **train**, e una delle sue caratteristiche è la probabilità media della rete dentro l'oggetto.
Nessun training: si caricano i pesi del fine-tuning già fatto.

Prima di eseguire:
1. **Settings → Accelerator → GPU T4 x2** e **Settings → Internet → On** (per installare Cellpose).
2. **Add Input**: i dataset `cellseg-code` e `fnc-green-data`, più l'**output del notebook Cellpose** di ieri
   (Add Input → Your Work / Notebooks → il notebook Cellpose → la versione completa): contiene i pesi `cellpose_ft_green`.

Uso: prova breve con `PROBE = True` (cella 3) → **Run All**; poi `PROBE = False` → **Save & Run All (Commit)** (~1.5 h).
Uscita: `results_cellpose_ftp.zip` (maschere + probabilità, niente pesi) → scaricala in `kaggle/` del progetto.
Le maschere e le probabilità di **test** si calcolano ora ma si valutano solo il giorno del test.

In [ ]:
# 1) Install Cellpose WITHOUT touching numpy and scipy (a change in the middle of a session breaks the imports)
import numpy
import scipy
NUMPY, SCIPY = numpy.__version__, scipy.__version__
print("keeping numpy", NUMPY, "and scipy", SCIPY)
!pip install -q cellpose "numpy=={NUMPY}" "scipy=={SCIPY}"
from importlib.metadata import version
CP_VERSION = version("cellpose")
print("cellpose", CP_VERSION)

In [ ]:
# 2) GPU available?
import torch
print("torch", torch.__version__)
assert torch.cuda.is_available(), "no GPU: Settings → Accelerator → 'GPU T4 x2'"
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 3) Data, split and the fine-tuned weights
import json
import time
from pathlib import Path

import numpy as np
import skimage.io as io

PROBE = False        # True = short check (2 images per split)

INPUT = Path("/kaggle/input")
OUT = Path("/kaggle/working/runs/cellpose" + ("_probe" if PROBE else ""))
OUT.mkdir(parents=True, exist_ok=True)

def find_one(pattern):
    '''First path under /kaggle/input matching the pattern (Kaggle may nest the dataset folders).'''
    hits = sorted(INPUT.rglob(pattern))
    assert hits, f"not found under {INPUT}: {pattern} (did you add the datasets and the Cellpose notebook output?)"
    return hits[0]

split = json.load(open(find_one("splits/green_split.json")))
TRAINVAL, TEST = find_one("green/trainval/images"), find_one("green/test/images")
WEIGHTS = find_one("cellpose_ft_green")          # fine-tuned weights from the Cellpose notebook output
SPLITS = {"train": (TRAINVAL, split["train"]), "val": (TRAINVAL, split["val"]),
          "test": (TEST, sorted(p.name for p in TEST.glob("*.png")))}
assert [len(names) for _, names in SPLITS.values()] == [169, 43, 70], "expected 169 / 43 / 70 images"
if PROBE:
    SPLITS = {s: (folder, names[:2]) for s, (folder, names) in SPLITS.items()}
print("weights:", WEIGHTS)
for s, (folder, names) in SPLITS.items():
    print(f"{s}: {len(names)} images from {folder}")

def signal(folder, name):
    '''The G channel (the c-FOS signal) as a 2-D uint8 image.'''
    return io.imread(folder / name)[..., 1]

In [ ]:
# 4) Fine-tuned model: instance masks + cell probability (sigmoid of Cellpose's cellprob), same settings as before
from cellpose import models

DIAMETER = 30
model = models.CellposeModel(gpu=True, pretrained_model=str(WEIGHTS))

def segment(split_name):
    '''Masks (uint16) and cell probability (uint8, probability × 255) of every image of a split; seconds per image.'''
    folder, names = SPLITS[split_name]
    masks, probs, start = [], [], time.time()
    for name in names:
        g = signal(folder, name)
        m, flows = model.eval(g, diameter=DIAMETER)[:2]
        cellprob = np.clip(np.asarray(flows[2], dtype=np.float32), -30, 30)   # beyond ±30 the sigmoid is already 0 or 1
        assert cellprob.shape == g.shape, f"unexpected cellprob shape {cellprob.shape}: flows = {[np.shape(f) for f in flows]}"
        masks.append(m.astype(np.uint16))
        probs.append(np.round(255 / (1 + np.exp(-cellprob))).astype(np.uint8))
    return np.stack(masks), np.stack(probs), (time.time() - start) / len(names)

timing = {}
for s in SPLITS:
    masks, probs, sec = segment(s)
    np.savez_compressed(OUT / f"cellpose_ftp_{s}.npz", masks=masks, probs=probs, names=np.array(SPLITS[s][1]))
    timing[f"{s}_s_per_image"] = sec
    inside = probs[masks > 0].mean() / 255 if (masks > 0).any() else float("nan")
    print(f"{s}: {len(masks)} images, {sec:.1f} s/image | mean probability inside objects {inside:.2f}, "
          f"outside {probs[masks == 0].mean() / 255:.3f}")

In [ ]:
# 5) Configuration + one zip with masks and probabilities only
import shutil

with open(OUT / "cellpose_ftp_config.json", "w") as f:
    json.dump({"cellpose_version": CP_VERSION, "weights": str(WEIGHTS), "diameter": DIAMETER,
               "probability": "sigmoid(cellprob) × 255, uint8", "timing": timing, "probe": PROBE}, f, indent=2)
shutil.make_archive("/kaggle/working/results_cellpose_ftp" + ("_probe" if PROBE else ""), "zip",
                    root_dir="/kaggle/working", base_dir=str(OUT.relative_to("/kaggle/working")))
print(sorted(p.name for p in OUT.iterdir()))